In [0]:
from pathlib import Path
from pyspark.sql.functions import current_timestamp, lit

catalogo = "will"
esquema = "i30"

pasta_dados = Path(
    "/Workspace/Users/willdeglan@gmail.com/Projetos/i30/datasource"
)

arquivos = {
    "viagens": "registro_viagens.csv",
    "caronas": "registro_caronas.csv",
    "abastecimentos": "registro_abastecimento.csv",
}

tabelas_bronze = {
    "viagens": "bronze_viagens",
    "caronas": "bronze_caronas",
    "abastecimentos": "bronze_abastecimento",
}

# Criando o catalogo e o esquema
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalogo}.{esquema}")

dados = {}

for nome, arquivo in arquivos.items():
    caminho = pasta_dados / arquivo

    if not caminho.is_file():
        raise FileNotFoundError(f"CSV não encontrado: {caminho}")

    dados[nome] = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "false")
        .option("encoding", "UTF-8")
        .csv(f"file:{caminho}")
    )

for nome, tabela in tabelas_bronze.items():
    df_bronze = (
        dados[nome]
        .withColumn("_ingested_at", current_timestamp())
        .withColumn("_source_file", lit(arquivos[nome]))
    )

    (
        df_bronze.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{catalogo}.{esquema}.{tabela}")
    )

    print(f"Tabela gravada: {catalogo}.{esquema}.{tabela}")

In [0]:
from pathlib import Path
from pyspark.sql.functions import current_timestamp, lit

catalogo = "will"
esquema = "i30"

for tabela in tabelas_bronze.values():
    nome_completo = f"{catalogo}.{esquema}.{tabela}"
    tabela_sql = f"`{catalogo}`.`{esquema}`.`{tabela}`"

    print(f"\n--- Dados: {nome_completo} ---")
    display(spark.table(nome_completo))

    print(f"\n--- Descrição e comentários: {nome_completo} ---")
    display(spark.sql(f"DESCRIBE TABLE EXTENDED {tabela_sql}"))